# Explainable AI-Based Duchenne Muscular Dystrophy Progression Prediction Using Machine Learning

**Data:** open KU Leuven Research Data Repository dataset *Strength, ROM and CSA in DMD* (https://doi.org/10.48804/ONRNG0), the data behind Vandekerckhove et al., *Longitudinal trajectories of muscle impairments in growing boys with Duchenne muscular dystrophy* (medRxiv 2024). Please cite both the dataset and the paper.

**Task:** from a boy's current visit (lower-limb strength, range of motion, age band, body size, mutation type, steroid), predict whether his **lower-limb muscle strength will decline by the next visit**, and explain every prediction with SHAP and LIME.

**Pipeline:** Setup -> Load and clean -> EDA -> Build the progression target -> Patient-level validation -> Models -> Evaluation -> Explainability -> Interpretation -> Limitations -> Export for Streamlit

> **Key design choice:** each boy has several visits, so a normal random train/test split would put the same boy in both sets and inflate the results. Everything here is split **by boy** (grouped cross-validation).

## 1. Setup

In [ ]:
# Uncomment on a fresh environment:
# !pip install -q scikit-learn xgboost shap lime seaborn pandas matplotlib joblib

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, average_precision_score, roc_auc_score,
                             roc_curve, precision_recall_curve, ConfusionMatrixDisplay)
from xgboost import XGBClassifier
import shap
from lime.lime_tabular import LimeTabularExplainer

SEED = 42
np.random.seed(SEED)
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 60)

## 2. Load and clean

The CSV has one row per boy per visit (`ID`, `Session`), with one leg measured per boy (`side`). The bottom of the file contains a legend table (abbreviations and category definitions) that is not data and must be removed.

In [ ]:
CSV_PATH = "Dataset_Strength_ROM_CSA_in_DMD.csv"

raw = pd.read_csv(CSV_PATH, encoding="latin-1", sep=";")    # semicolon-separated, not UTF-8
raw.columns = raw.columns.str.strip()

# keep only real data rows (numeric ID and Session); the legend rows below the data are dropped
is_data = pd.to_numeric(raw["ID"], errors="coerce").notna() & raw["Session"].notna()
df = raw[is_data].copy()
df["ID"] = df["ID"].astype(int)
df["Session"] = df["Session"].astype(int)

for c in ["Mutation Type", "Corticosteroids", "side"]:
    df[c] = df[c].astype("string").str.strip()
df["Corticosteroids"] = df["Corticosteroids"].fillna("None")

STRENGTH = [c for c in df.columns if c.startswith("Strength") and c.endswith("_z")]
ROM = [c for c in df.columns if c.startswith("ROM") and c.endswith("_z")]
CSA = [c for c in df.columns if c.startswith("CSA") and c.endswith("_z")]

print(f"{df.ID.nunique()} boys, {len(df)} visits, {df.groupby('ID').size().median():.0f} visits per boy (median)")
print("Strength z columns:", STRENGTH)
print("ROM z columns     :", ROM)
df.head()

## 3. Exploratory data analysis

In [ ]:
AGE_LABELS = {1: "<6 y", 2: "6-9 y", 3: "9-12 y", 4: "12-15 y", 5: ">15 y"}

df["Strength_composite_z"] = df[STRENGTH].mean(axis=1).where(df[STRENGTH].notna().sum(axis=1) >= 5)
df["ROM_composite_z"] = df[ROM].mean(axis=1).where(df[ROM].notna().sum(axis=1) >= 4)

fig, ax = plt.subplots(2, 2, figsize=(14, 9))

sns.countplot(x=df.groupby("ID").size(), ax=ax[0, 0], color="#2a7f62")
ax[0, 0].set_title("Number of visits per boy"); ax[0, 0].set_xlabel("visits")

zlong = df[STRENGTH + ROM].melt(var_name="measure", value_name="z").dropna()
zlong["measure"] = zlong["measure"].str.replace("_z", "")
sns.boxplot(data=zlong, y="measure", x="z", ax=ax[0, 1], color="#9ecae1")
ax[0, 1].axvline(0, color="k", lw=1); ax[0, 1].set_title("Z-scores vs typically developing peers (0 = typical)")

for pid, g in df.dropna(subset=["Strength_composite_z"]).groupby("ID"):
    ax[1, 0].plot(g["Age_category"] + np.random.uniform(-.1, .1, len(g)), g["Strength_composite_z"], alpha=.35, lw=1)
sns.pointplot(data=df, x="Age_category", y="Strength_composite_z", color="k", ax=ax[1, 0], errorbar="se")
ax[1, 0].set_xticklabels([AGE_LABELS.get(int(float(t.get_text())), t.get_text()) for t in ax[1, 0].get_xticklabels()])
ax[1, 0].set_title("Composite strength z by age band (lines = individual boys)")

sns.heatmap(df[STRENGTH + ROM].corr(), cmap="coolwarm", center=0, ax=ax[1, 1], cbar_kws={"shrink": .8})
ax[1, 1].set_title("Correlation of strength and ROM z-scores")
plt.tight_layout(); plt.show()

print("Mutation type :", df.drop_duplicates("ID")["Mutation Type"].value_counts().to_dict())
print("Corticosteroid:", df["Corticosteroids"].value_counts().to_dict())
print("Missing % (z-scores):")
display((df[STRENGTH + ROM + CSA].isna().mean() * 100).round(0).to_frame("% missing").T)

## 4. Build the progression target

For every boy, consecutive visits form a **visit pair** (visit *t* -> visit *t+1*). The features come from visit *t* and the label says whether strength got worse by visit *t+1*:

- **Composite strength z** = mean of the 7 lower-limb strength z-scores (needs at least 5 measured).
- **Strength decline (1)** if the composite z fell by at least `DECLINE_THRESHOLD` (default 0.25 z) at the next visit, otherwise **stable or improving (0)**.

The dataset does not contain visit dates, so the time between visits is unknown (a limitation, discussed at the end). Choose the threshold together with your supervisor; the default gives a reasonably balanced outcome.

In [ ]:
DECLINE_THRESHOLD = 0.25
CLASS_NAMES = ["Stable or improving strength", "Strength decline"]

df = df.sort_values(["ID", "Session"]).reset_index(drop=True)
g = df.groupby("ID")
df["next_strength_z"] = g["Strength_composite_z"].shift(-1)
df["delta_next"] = df["next_strength_z"] - df["Strength_composite_z"]
df["prev_change"] = df["Strength_composite_z"] - g["Strength_composite_z"].shift(1)   # change since the previous visit

pairs = df.dropna(subset=["delta_next"]).copy()
pairs["y"] = (pairs["delta_next"] <= -DECLINE_THRESHOLD).astype(int)

print(f"{len(pairs)} visit pairs from {pairs.ID.nunique()} boys")
print("Outcome:", {CLASS_NAMES[k]: int(v) for k, v in pairs.y.value_counts().sort_index().items()})

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
sns.histplot(pairs["delta_next"], bins=25, ax=ax[0], color="#2a7f62")
ax[0].axvline(-DECLINE_THRESHOLD, color="r", ls="--"); ax[0].set_title("Change in composite strength z to next visit")
sns.countplot(x=pairs["y"].map(dict(enumerate(CLASS_NAMES))), ax=ax[1]); ax[1].set_xlabel(""); ax[1].set_title("Outcome")
plt.tight_layout(); plt.show()

## 5. Features and patient-level validation

**Features (all known at the current visit):** the 7 strength z-scores, the 6 range-of-motion z-scores, the two composites, age band, body-mass / height / BMI bands, visit number, change since the previous visit, mutation type and corticosteroid.

The next-visit values (`next_strength_z`, `delta_next`) are **never** used as features. Cross-sectional area (CSA) is left out because 70-80% of it is missing.

In [ ]:
num_cols = (STRENGTH + ROM + ["Strength_composite_z", "ROM_composite_z", "Age_category",
            "BodyMass_category", "Height_category", "BMI_category", "Session", "prev_change"])
cat_cols = ["Mutation Type", "Corticosteroids"]

X = pairs[num_cols + cat_cols].reset_index(drop=True)
for c in cat_cols:
    X[c] = X[c].astype(object)
y = pairs["y"].values
groups = pairs["ID"].values
print("X:", X.shape, "| positives:", y.sum(), f"({y.mean():.0%})")

def make_preprocessor(scale=False):
    num_steps = [("imp", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("sc", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_steps), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols),
    ])

# name -> (factory building a fresh pipeline, uses sample weights?)
factories = {
    "Baseline (prior)": (lambda: Pipeline([("pre", make_preprocessor()), ("clf", DummyClassifier(strategy="prior"))]), False),
    "Logistic Regression": (lambda: Pipeline([("pre", make_preprocessor(scale=True)),
                            ("clf", LogisticRegression(C=0.3, max_iter=3000, class_weight="balanced", random_state=SEED))]), False),
    "Random Forest": (lambda: Pipeline([("pre", make_preprocessor()),
                      ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced",
                                                     n_jobs=-1, random_state=SEED))]), False),
    "XGBoost": (lambda: Pipeline([("pre", make_preprocessor()),
                ("clf", XGBClassifier(n_estimators=150, max_depth=2, learning_rate=0.05, subsample=0.8,
                                      colsample_bytree=0.8, min_child_weight=3, reg_lambda=5.0,
                                      eval_metric="logloss", n_jobs=-1, random_state=SEED))]), True),
}

def fit_model(factory_entry, Xtr, ytr):
    make, weighted = factory_entry
    m = make()
    if weighted:
        m.fit(Xtr, ytr, clf__sample_weight=compute_sample_weight("balanced", ytr))
    else:
        m.fit(Xtr, ytr)
    return m

def grouped_cv_oof(entry, X, y, groups, seed):
    # out-of-fold probabilities; the same boy is never in train and test together
    oof = np.zeros(len(y))
    for tr, te in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed).split(X, y, groups):
        m = fit_model(entry, X.iloc[tr], y[tr])
        oof[te] = m.predict_proba(X.iloc[te])[:, 1]
    return oof

## 6. Evaluation (repeated patient-level cross-validation)

With only ~100 pairs, a single train/test split would be very noisy. Instead we repeat 5-fold cross-validation **by boy** 10 times with different shuffles and report the mean and spread. `Baseline (prior)` always predicts the average risk, so a useful model must beat its ROC-AUC of 0.5 and its PR-AUC of roughly the outcome rate.

In [ ]:
N_REPEATS = 10
rows, oof_store = [], {}

for name, entry in factories.items():
    per_seed = []
    for s in range(N_REPEATS):
        oof = grouped_cv_oof(entry, X, y, groups, seed=SEED + s)
        if s == 0:
            oof_store[name] = oof
        pred = (oof >= 0.5).astype(int)
        per_seed.append({
            "ROC-AUC": roc_auc_score(y, oof), "PR-AUC": average_precision_score(y, oof),
            "Balanced Acc": balanced_accuracy_score(y, pred), "Precision": precision_score(y, pred, zero_division=0),
            "Recall": recall_score(y, pred, zero_division=0), "F1": f1_score(y, pred, zero_division=0),
            "MCC": matthews_corrcoef(y, pred)})
    r = pd.DataFrame(per_seed)
    rows.append({"Model": name, **r.mean().to_dict(), "ROC-AUC sd": r["ROC-AUC"].std()})

results = pd.DataFrame(rows).set_index("Model").round(3).sort_values("ROC-AUC", ascending=False)
print(f"Outcome rate (PR-AUC of a useless model): {y.mean():.2f}")
display(results.style.background_gradient(cmap="Greens", subset=["ROC-AUC", "PR-AUC", "MCC"]))

best_name = results.drop(index="Baseline (prior)").index[0]
print("Best model by ROC-AUC:", best_name)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
for name, oof in oof_store.items():
    fpr, tpr, _ = roc_curve(y, oof); ax[0].plot(fpr, tpr, label=f"{name} ({roc_auc_score(y, oof):.2f})")
    pr, rc, _ = precision_recall_curve(y, oof); ax[1].plot(rc, pr, label=name)
ax[0].plot([0, 1], [0, 1], "k--", lw=1); ax[0].set_title("ROC (out-of-fold, seed 0)"); ax[0].legend(fontsize=8)
ax[0].set_xlabel("False positive rate"); ax[0].set_ylabel("True positive rate")
ax[1].axhline(y.mean(), color="k", ls="--", lw=1); ax[1].set_title("Precision-recall"); ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision")
ConfusionMatrixDisplay.from_predictions(y, (oof_store[best_name] >= 0.5).astype(int), display_labels=["Stable", "Decline"],
                                        normalize="true", values_format=".2f", ax=ax[2], colorbar=False)
ax[2].set_title(f"Confusion matrix - {best_name}")
plt.tight_layout(); plt.show()

### 6.1 Is the signal real? Confidence interval and permutation test

Because the sample is small, we check (a) a bootstrap 95% confidence interval for the ROC-AUC (resampling **boys**, not visits) and (b) a permutation test: how often would shuffled labels reach the observed AUC by chance?

In [ ]:
oof_best = oof_store[best_name]
obs_auc = roc_auc_score(y, oof_best)

# (a) bootstrap over boys
rng = np.random.RandomState(SEED)
ids = np.unique(groups)
idx_by_id = {i: np.where(groups == i)[0] for i in ids}
boot = []
for _ in range(1000):
    pick = np.concatenate([idx_by_id[i] for i in rng.choice(ids, size=len(ids), replace=True)])
    if len(np.unique(y[pick])) == 2:
        boot.append(roc_auc_score(y[pick], oof_best[pick]))
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"{best_name}: ROC-AUC = {obs_auc:.3f}  (95% CI over boys: {lo:.3f} - {hi:.3f})")

# (b) permutation test
N_PERM = 100
null = []
for _ in range(N_PERM):
    y_perm = rng.permutation(y)
    null.append(roc_auc_score(y_perm, grouped_cv_oof(factories[best_name], X, y_perm, groups, seed=SEED)))
p_value = (1 + np.sum(np.array(null) >= obs_auc)) / (N_PERM + 1)
print(f"Permutation p-value = {p_value:.3f}  (null AUC mean {np.mean(null):.3f})")

plt.figure(figsize=(6, 3.4))
sns.histplot(null, bins=20, color="grey"); plt.axvline(obs_auc, color="r", label="observed")
plt.title("AUC under shuffled labels"); plt.legend(); plt.tight_layout(); plt.show()

> **How to read these numbers.** In the run used to prepare this notebook, Logistic Regression reached a mean ROC-AUC of about 0.60 over the repeated runs, while Random Forest and XGBoost stayed near 0.5 (chance). The single-run confidence interval includes 0.5 and the permutation p-value is around 0.2, so the evidence that visit-to-visit strength decline can be predicted from these variables is **weak**. Report it that way: a small, honest, explainable pipeline, not a validated predictor.

## 7. Explainability (XAI)

Explanations are produced for the best model, fitted on all visit pairs (the cross-validation above already gives the honest performance estimate). SHAP values are for the class **Strength decline**: a positive value pushes the boy toward decline. Linear models use SHAP's `LinearExplainer`, tree models use `TreeExplainer`.

In [ ]:
xai_name = best_name
xai_pipe = fit_model(factories[xai_name], X, y)
pre, clf = xai_pipe.named_steps["pre"], xai_pipe.named_steps["clf"]
print("Explaining:", xai_name)

feat_names = [n.split("__", 1)[1] for n in pre.get_feature_names_out()]
X_t = pre.transform(X)

def display_matrix(pre, Xdf):
    # imputed but NOT scaled numeric values + one-hot categories: readable units for plots
    num_part = pre.named_transformers_["num"].named_steps["imp"].transform(Xdf[num_cols])
    cat_part = pre.named_transformers_["cat"].transform(Xdf[cat_cols])
    return np.hstack([num_part, cat_part])

disp = pd.DataFrame(display_matrix(pre, X), columns=feat_names)

explainer = (shap.LinearExplainer(clf, shap.maskers.Independent(X_t, max_samples=len(X_t)))
             if hasattr(clf, "coef_") else shap.TreeExplainer(clf))
sv = explainer.shap_values(X_t) if hasattr(clf, "coef_") else explainer.shap_values(X_t, check_additivity=False)
sv = np.stack(sv, axis=-1) if isinstance(sv, list) else np.asarray(sv)
base = np.atleast_1d(explainer.expected_value)
if sv.ndim == 2:                         # models that return values for the positive class only
    sv = np.stack([-sv, sv], axis=-1)
    base = np.array([-base[0], base[0]])
POS = 1
print("SHAP array shape (pairs, features, classes):", sv.shape)

### 7.1 Global explanations

In [ ]:
global_imp = pd.Series(np.abs(sv[:, :, POS]).mean(axis=0), index=feat_names).sort_values(ascending=False)

def original_col(feature):
    for c in cat_cols:
        if feature.startswith(c + "_"):
            return c
    return feature

grouped = global_imp.groupby(original_col).sum().sort_values(ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
global_imp.head(15)[::-1].plot.barh(ax=ax[0], color="#2a7f62")
ax[0].set_title(f"Top 15 features - {xai_name}"); ax[0].set_xlabel("mean |SHAP value|")
grouped.head(12)[::-1].plot.barh(ax=ax[1], color="#1f5f8b")
ax[1].set_title("Top 12 original variables (one-hot merged)"); ax[1].set_xlabel("summed mean |SHAP value|")
plt.tight_layout(); plt.show()

In [ ]:
shap.summary_plot(sv[:, :, POS], disp, max_display=15, show=False)
plt.title("SHAP beeswarm - effect on risk of strength DECLINE (red = high feature value)")
plt.tight_layout(); plt.show()

top_num = [f for f in grouped.index if f in num_cols][:2]
for f in top_num:
    shap.dependence_plot(f, sv[:, :, POS], disp, show=False)
    plt.title(f"Dependence plot: {f}")
    plt.tight_layout(); plt.show()

### 7.2 Local explanations (SHAP waterfall + LIME)

In [ ]:
proba_all = xai_pipe.predict_proba(X)[:, 1]
cases = {"highest predicted risk": int(np.argmax(proba_all)), "lowest predicted risk": int(np.argmin(proba_all))}

for title, i in cases.items():
    print(f"Boy {pairs.iloc[i]['ID']} at visit {pairs.iloc[i]['Session']}: predicted probability of decline = {proba_all[i]:.0%}"
          f" | actual outcome: {CLASS_NAMES[y[i]]}")
    shap.plots.waterfall(shap.Explanation(values=sv[i, :, POS], base_values=base[POS],
                                          data=disp.iloc[i].values, feature_names=feat_names),
                         max_display=12, show=False)
    plt.title(f"SHAP waterfall - {title}")
    plt.tight_layout(); plt.show()

In [ ]:
lime_explainer = LimeTabularExplainer(X_t, feature_names=feat_names, class_names=CLASS_NAMES,
                                      mode="classification", discretize_continuous=True, random_state=SEED)
i = cases["highest predicted risk"]
exp = lime_explainer.explain_instance(X_t[i], clf.predict_proba, num_features=10, labels=(POS,))
exp.as_pyplot_figure(label=POS)
plt.title("LIME - same boy/visit as the first SHAP waterfall (features shown in model units)")
plt.tight_layout(); plt.show()

top_shap = pd.Series(sv[i, :, POS], index=feat_names).abs().sort_values(ascending=False).head(10).index.tolist()
top_lime = [f for f, _ in exp.as_list(label=POS)]
print("Top SHAP features :", top_shap)
print("Top LIME features :", top_lime)

## 8. Interpretation

Fill in from your own plots:

- **Performance:** quote the ROC-AUC with its 95% CI and the permutation p-value from section 6.1, and compare with the baseline. Be honest if the signal is modest.
- **Main drivers:** list the top 5 variables from 7.1 and group them (strength deficits, contractures/ROM, growth stage, treatment, mutation type).
- **Direction of effect:** use the beeswarm and dependence plots. Does a higher current strength z (less deficit) raise or lower the chance of a further decline? Note that boys who are already very weak have less room to decline (a floor effect), and strong boys can regress toward the mean.
- **SHAP vs LIME:** how many of the top 10 features overlap?
- **Clinical relevance:** compare with the paper's findings (strength and plantar-flexion ROM show the steepest deficits; many outcomes are stable or improve until about 6.6-9.5 years and decline afterwards).

## 9. Limitations

- **Small sample:** 33 boys and about 100 visit pairs. Confidence intervals are wide and results can change with a different set of boys.
- **Unknown time between visits:** the dataset has no visit dates, only a visit number and an age band, so "next visit" covers different time spans for different boys.
- **Coarse age:** age is given in 3-year bands, which blurs the age effect that matters most in DMD.
- **Measurement noise:** strength z-scores from hand-held/fixed dynamometry fluctuate between sessions, so some "declines" are noise.
- **Not the whole disease:** only lower-limb strength, range of motion and (mostly missing) muscle size; no NSAA, 6MWT, loss-of-ambulation or genotype severity beyond mutation type and exon.
- **Explanations are not causes:** SHAP and LIME show what the model uses, not biology.
- **Ethics:** research prototype only, not a clinical decision aid. Cite the dataset and paper authors.

**Future work:** time-aware models using exact ages and visit dates, mixed-effects trajectory models as a clinically standard comparison, combining with larger registries, and external validation.

## 10. Export for the Streamlit app

Saves the fitted model and everything the app needs into `dmd_model.joblib`. Then run `streamlit run app.py` in the same folder.

In [ ]:
import joblib

MUSCLE = {"HF": "Hip flexion", "HAB": "Hip abduction", "HE": "Hip extension", "KE": "Knee extension",
          "KF": "Knee flexion", "DF": "Dorsiflexion", "PF": "Plantar flexion", "HAD": "Hip adduction",
          "Popl": "Popliteal angle", "DFwithKE": "Dorsiflexion (knee extended)", "DFwithKF": "Dorsiflexion (knee flexed)"}
pretty = {}
for c in STRENGTH:
    pretty[c] = "Strength - " + MUSCLE[c.split("_")[1]] + " (z)"
for c in ROM:
    pretty[c] = "ROM - " + MUSCLE[c.split("_")[1]] + " (z)"
pretty.update({"Session": "Visit number", "prev_change": "Change in composite strength z since previous visit",
               "Age_category": "Age band", "BodyMass_category": "Body mass band",
               "Height_category": "Height band", "BMI_category": "BMI band"})

ordinal_labels = {
    "Age_category": {1: "<6 years", 2: "6-9 years", 3: "9-12 years", 4: "12-15 years", 5: ">15 years"},
    "BodyMass_category": {1: "<20 kg", 2: "20-30 kg", 3: "30-40 kg", 4: "40-50 kg", 5: ">50 kg"},
    "Height_category": {1: "<1.10 m", 2: "1.10-1.20 m", 3: "1.20-1.30 m", 4: "1.30-1.40 m", 5: ">1.40 m"},
    "BMI_category": {1: "<18.5", 2: "18.5-25", 3: "25-30", 4: ">30"},
}

art = {
    "title": "Explainable DMD Progression Prediction (lower-limb strength)",
    "pipe": xai_pipe,
    "model_name": xai_name,
    "class_names": CLASS_NAMES,
    "pos_index": POS,
    "decline_threshold": DECLINE_THRESHOLD,
    "num_cols": num_cols, "cat_cols": cat_cols,
    "strength_cols": STRENGTH, "rom_cols": ROM,
    "num_stats": {c: {"min": float(X[c].min()), "max": float(X[c].max()), "median": float(X[c].median())} for c in num_cols},
    "cat_options": {c: sorted(X[c].dropna().astype(str).unique().tolist()) for c in cat_cols},
    "ordinal_labels": ordinal_labels, "pretty": pretty,
    "feat_names": feat_names,
    "background": X_t,
    "global_importance": grouped.head(12).to_dict(),
    "results": results.reset_index().to_dict("records"),
    "cv_note": f"Mean ROC-AUC over {N_REPEATS} repeated patient-level CV runs: {results.loc[best_name, 'ROC-AUC']:.2f}. "
               f"Single run: {obs_auc:.2f} (95% CI over boys {lo:.2f}-{hi:.2f}), permutation p = {p_value:.2f}. "
               f"{len(X)} visit pairs from {len(ids)} boys; outcome rate {y.mean():.0%}. "
               "A signal this small is not clearly separable from chance.",
}
joblib.dump(art, "dmd_model.joblib")
print("Saved dmd_model.joblib  ->  now run:  streamlit run app.py")